# Exploratory Data Analysis — Smart MCQ Solver
**Roll No**: 23f3002157 | **Project**: BSDA2001P

Covers: distributions, text statistics, similarity analysis, hypothesis testing, class balance.

In [1]:
import os
import sys
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, shapiro
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

plt.rcParams.update({
    'figure.dpi'       : 130,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.size'        : 11,
})
PALETTE     = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']
OPTION_COLS = ['A','B','C','D','E']

df = pd.read_csv('../data/raw/train.csv')
os.makedirs('../reports', exist_ok=True)
print(f'Shape : {df.shape}')
df.head(3)

Shape : (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


---
## 1. Dataset Overview

In [2]:
print('=== Basic Info ===')
df.info()
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'\nAnswer distribution:\n{df["answer"].value_counts()}')
print(f'\nAnswer proportions:\n{df["answer"].value_counts(normalize=True).round(4)}')

=== Basic Info ===
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2000 non-null   int64
 1   prompt  2000 non-null   str  
 2   A       2000 non-null   str  
 3   B       2000 non-null   str  
 4   C       2000 non-null   str  
 5   D       2000 non-null   str  
 6   E       2000 non-null   str  
 7   answer  2000 non-null   str  
dtypes: int64(1), str(7)
memory usage: 1.9 MB

Missing values:
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Duplicate rows: 0

Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Answer proportions:
answer
B    0.2450
C    0.2295
A    0.1845
D    0.1790
E    0.1620
Name: proportion, dtype: float64


---
## 2. Answer Distribution

In [ ]:
freq       = df['answer'].value_counts()
freq_pct   = df['answer'].value_counts(normalize=True) * 100
uniform_pct = 100 / 5

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bars = axes[0].bar(freq.index, freq.values, color=PALETTE, edgecolor='black', linewidth=0.6)
axes[0].axhline(len(df)/5, color='red', linestyle='--', linewidth=1.2, label='Expected (uniform)')
axes[0].set_title('Correct Answer Frequency')
axes[0].set_xlabel('Answer Option')
axes[0].set_ylabel('Count')
axes[0].legend()
for bar, val in zip(bars, freq.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontsize=10)

axes[1].pie(freq.values, labels=freq.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Answer Distribution (Pie)')

plt.tight_layout()
plt.savefig('../reports/eda_answer_distribution.png', bbox_inches='tight')
plt.show()

---
## 3. Text Length Analysis

In [ ]:
def word_count(text): return len(str(text).split())
def char_count(text): return len(str(text))

df['prompt_words'] = df['prompt'].apply(word_count)
df['prompt_chars'] = df['prompt'].apply(char_count)

for col in OPTION_COLS:
    df[f'{col}_words'] = df[col].apply(word_count)
    df[f'{col}_chars'] = df[col].apply(char_count)

df['mean_option_words'] = df[[f'{c}_words' for c in OPTION_COLS]].mean(axis=1)
df['correct_opt_words'] = df.apply(lambda r: word_count(r[r['answer']]), axis=1)

print('=== Prompt Length Stats (words) ===')
print(df['prompt_words'].describe().round(2))
print('\n=== Option Length Stats (words) ===')
for col in OPTION_COLS:
    s = df[f'{col}_words']
    print(f'  {col}: mean={s.mean():.1f}  std={s.std():.1f}  min={s.min()}  max={s.max()}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Prompt word count distribution
axes[0,0].hist(df['prompt_words'], bins=40, color='#4C72B0', edgecolor='white', linewidth=0.4)
axes[0,0].axvline(df['prompt_words'].mean(), color='red', linestyle='--', label=f'Mean={df["prompt_words"].mean():.1f}')
axes[0,0].axvline(df['prompt_words'].median(), color='orange', linestyle='--', label=f'Median={df["prompt_words"].median():.1f}')
axes[0,0].set_title('Prompt Word Count Distribution')
axes[0,0].set_xlabel('Word Count')
axes[0,0].legend(fontsize=9)

# Option word count by option label
opt_lengths = pd.DataFrame({col: df[f'{col}_words'] for col in OPTION_COLS})
axes[0,1].boxplot([opt_lengths[c] for c in OPTION_COLS], labels=OPTION_COLS,
                  patch_artist=True,
                  boxprops=dict(facecolor='#AEC6CF'),
                  medianprops=dict(color='red', linewidth=2))
axes[0,1].set_title('Option Word Count Distribution')
axes[0,1].set_xlabel('Option')
axes[0,1].set_ylabel('Word Count')

# Correct option word count vs incorrect
correct_lengths   = df['correct_opt_words'].values
incorrect_lengths = []
for _, row in df.iterrows():
    for col in OPTION_COLS:
        if col != row['answer']:
            incorrect_lengths.append(word_count(row[col]))

axes[1,0].hist(correct_lengths,   bins=40, alpha=0.6, color='green', label='Correct option',   edgecolor='white')
axes[1,0].hist(incorrect_lengths, bins=40, alpha=0.4, color='red',   label='Incorrect options', edgecolor='white')
axes[1,0].set_title('Correct vs Incorrect Option Length')
axes[1,0].set_xlabel('Word Count')
axes[1,0].legend(fontsize=9)

# Prompt length by correct answer
groups = [df[df['answer']==c]['prompt_words'].values for c in OPTION_COLS]
axes[1,1].boxplot(groups, labels=OPTION_COLS, patch_artist=True,
                  boxprops=dict(facecolor='#FFD700'),
                  medianprops=dict(color='red', linewidth=2))
axes[1,1].set_title('Prompt Length by Correct Answer')
axes[1,1].set_xlabel('Correct Answer')
axes[1,1].set_ylabel('Prompt Word Count')

plt.tight_layout()
plt.savefig('../reports/eda_text_lengths.png', bbox_inches='tight')
plt.show()

---
## 4. Vocabulary & Token Analysis

In [ ]:
def clean(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('','', string.punctuation))
    return text.split()

prompt_tokens = []
option_tokens = []
for _, row in df.iterrows():
    prompt_tokens.extend(clean(row['prompt']))
    for col in OPTION_COLS:
        option_tokens.extend(clean(row[col]))

prompt_vocab  = set(prompt_tokens)
option_vocab  = set(option_tokens)
overlap_vocab = prompt_vocab & option_vocab

print(f'Prompt vocab size   : {len(prompt_vocab):,}')
print(f'Option vocab size   : {len(option_vocab):,}')
print(f'Vocab overlap       : {len(overlap_vocab):,} ({len(overlap_vocab)/len(prompt_vocab)*100:.1f}% of prompt vocab)')

prompt_freq = Counter(prompt_tokens)
top20       = prompt_freq.most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

words, counts = zip(*top20)
axes[0].barh(list(words)[::-1], list(counts)[::-1], color='#4C72B0', edgecolor='white')
axes[0].set_title('Top 20 Words in Prompts')
axes[0].set_xlabel('Frequency')

ranks   = np.arange(1, 101)
freqs   = [prompt_freq.most_common(100)[i][1] for i in range(100)]
axes[1].loglog(ranks, freqs, color='#DD8452', linewidth=2)
axes[1].set_title("Zipf's Law — Prompt Token Frequency")
axes[1].set_xlabel('Rank (log)')
axes[1].set_ylabel('Frequency (log)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/eda_vocabulary.png', bbox_inches='tight')
plt.show()

---
## 5. TF-IDF Cosine Similarity Analysis

In [ ]:
all_texts = df['prompt'].tolist()
for col in OPTION_COLS:
    all_texts.extend(df[col].tolist())

vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
vectorizer.fit(all_texts)

sim_data = {col: [] for col in OPTION_COLS}
correct_sims   = []
incorrect_sims = []

for _, row in df.iterrows():
    p_vec = vectorizer.transform([str(row['prompt'])])
    for col in OPTION_COLS:
        o_vec = vectorizer.transform([str(row[col])])
        sim   = float(cosine_similarity(p_vec, o_vec)[0][0])
        sim_data[col].append(sim)
        if col == row['answer']:
            correct_sims.append(sim)
        else:
            incorrect_sims.append(sim)

print('=== TF-IDF Cosine Similarity Stats ===')
for col in OPTION_COLS:
    arr = np.array(sim_data[col])
    print(f'  Option {col}: mean={arr.mean():.4f}  std={arr.std():.4f}  max={arr.max():.4f}')
print(f'\nCorrect option mean sim  : {np.mean(correct_sims):.4f}')
print(f'Incorrect option mean sim: {np.mean(incorrect_sims):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for col, color in zip(OPTION_COLS, PALETTE):
    axes[0].hist(sim_data[col], bins=40, alpha=0.5, label=col, color=color, edgecolor='none')
axes[0].set_title('TF-IDF Cosine Similarity: Prompt vs Each Option')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(correct_sims,   bins=40, alpha=0.6, color='green', label='Correct',   edgecolor='none')
axes[1].hist(incorrect_sims, bins=40, alpha=0.4, color='red',   label='Incorrect', edgecolor='none')
axes[1].axvline(np.mean(correct_sims),   color='darkgreen', linestyle='--', linewidth=1.5)
axes[1].axvline(np.mean(incorrect_sims), color='darkred',   linestyle='--', linewidth=1.5)
axes[1].set_title('Cosine Similarity: Correct vs Incorrect Options')
axes[1].set_xlabel('Cosine Similarity')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/eda_cosine_similarity.png', bbox_inches='tight')
plt.show()

---
## 6. Correlation Analysis

In [ ]:
numeric_cols = ['prompt_words', 'prompt_chars',
                'A_words','B_words','C_words','D_words','E_words',
                'mean_option_words', 'correct_opt_words']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Text Length Features')
plt.tight_layout()
plt.savefig('../reports/eda_correlation.png', bbox_inches='tight')
plt.show()

---
## 7. Hypothesis Testing

In [ ]:
print('=' * 60)
print('  HYPOTHESIS TEST 1: Chi-Square Goodness of Fit')
print('  H0: Answer options are uniformly distributed')
print('  H1: Answer distribution is non-uniform')
print('=' * 60)

observed  = np.array([freq[c] for c in OPTION_COLS])
expected  = np.full(5, len(df) / 5)
chi2, p   = stats.chisquare(observed, expected)

print(f'Observed counts  : {dict(zip(OPTION_COLS, observed))}')
print(f'Expected (uniform): {int(expected[0])} per class')
print(f'Chi2 statistic   : {chi2:.4f}')
print(f'p-value          : {p:.6f}')
print(f'Result           : {"REJECT H0 — distribution is non-uniform" if p < 0.05 else "FAIL TO REJECT H0"}')

In [ ]:
print('=' * 60)
print('  HYPOTHESIS TEST 2: Kruskal-Wallis Test')
print('  H0: Prompt length is the same across all correct answer groups')
print('  H1: At least one group has a different prompt length distribution')
print('=' * 60)

groups  = [df[df['answer'] == c]['prompt_words'].values for c in OPTION_COLS]
stat, p = kruskal(*groups)

print(f'Group means:')
for c, g in zip(OPTION_COLS, groups):
    print(f'  Answer={c}: mean={g.mean():.2f}  std={g.std():.2f}  n={len(g)}')
print(f'Kruskal-Wallis H : {stat:.4f}')
print(f'p-value          : {p:.6f}')
print(f'Result           : {"REJECT H0 — prompt length differs by answer group" if p < 0.05 else "FAIL TO REJECT H0"}')

In [ ]:
print('=' * 60)
print('  HYPOTHESIS TEST 3: Mann-Whitney U Test')
print('  H0: Correct and incorrect options have the same length distribution')
print('  H1: Correct options have a different length distribution')
print('=' * 60)

stat, p = mannwhitneyu(correct_lengths, incorrect_lengths, alternative='two-sided')

print(f'Correct option mean length  : {np.mean(correct_lengths):.2f}')
print(f'Incorrect option mean length: {np.mean(incorrect_lengths):.2f}')
print(f'Mann-Whitney U statistic    : {stat:.2f}')
print(f'p-value                     : {p:.6f}')
print(f'Result                      : {"REJECT H0 — correct options differ in length" if p < 0.05 else "FAIL TO REJECT H0"}')

In [ ]:
print('=' * 60)
print('  HYPOTHESIS TEST 4: Mann-Whitney U — TF-IDF Similarity')
print('  H0: Cosine similarity is the same for correct vs incorrect options')
print('  H1: Correct options have higher cosine similarity with the prompt')
print('=' * 60)

stat, p = mannwhitneyu(correct_sims, incorrect_sims, alternative='greater')

print(f'Correct mean similarity  : {np.mean(correct_sims):.4f}')
print(f'Incorrect mean similarity: {np.mean(incorrect_sims):.4f}')
print(f'Mann-Whitney U statistic : {stat:.2f}')
print(f'p-value (one-tailed)     : {p:.6f}')
print(f'Result                   : {"REJECT H0 — correct options are more similar to prompts" if p < 0.05 else "FAIL TO REJECT H0"}')

In [ ]:
print('=' * 60)
print('  HYPOTHESIS TEST 5: Chi-Square — Option Length vs Correct Answer')
print('  H0: Correct answer label is independent of correct option length bin')
print('  H1: Correct answer label depends on option length')
print('=' * 60)

df['correct_len_bin'] = pd.cut(df['correct_opt_words'],
                                bins=[0, 30, 60, 90, 999],
                                labels=['short','medium','long','very_long'])
contingency = pd.crosstab(df['answer'], df['correct_len_bin'])
print(f'Contingency table:\n{contingency}')

chi2, p, dof, expected_ct = chi2_contingency(contingency)
print(f'\nChi2 statistic : {chi2:.4f}')
print(f'Degrees of freedom: {dof}')
print(f'p-value        : {p:.6f}')
print(f'Result         : {"REJECT H0 — answer label depends on option length" if p < 0.05 else "FAIL TO REJECT H0"}')

In [ ]:
print('=' * 60)
print('  HYPOTHESIS TEST 6: Shapiro-Wilk Normality Test')
print('  H0: Prompt word counts are normally distributed')
print('  H1: Prompt word counts are not normally distributed')
print('=' * 60)

sample    = df['prompt_words'].sample(500, random_state=42).values
stat, p   = shapiro(sample)

print(f'Sample size    : 500')
print(f'Shapiro-Wilk W : {stat:.4f}')
print(f'p-value        : {p:.6f}')
print(f'Result         : {"REJECT H0 — prompt lengths are NOT normally distributed" if p < 0.05 else "FAIL TO REJECT H0"}')

---
## 8. Hypothesis Testing Summary Plot

In [ ]:
tests = [
    ('Chi-Sq: Answer uniformity',         'chisquare',   df['answer'].value_counts().values, np.full(5, len(df)/5)),
]

# Visualise p-values for all 6 tests
test_names = [
    'Chi-Sq: Answer uniformity',
    'Kruskal-Wallis: Prompt len by group',
    'Mann-Whitney: Option length',
    'Mann-Whitney: TF-IDF similarity',
    'Chi-Sq: Answer vs option length',
    'Shapiro-Wilk: Prompt normality',
]
p_values = []
p_values.append(stats.chisquare(observed, expected)[1])
p_values.append(kruskal(*groups)[1])
p_values.append(mannwhitneyu(correct_lengths, incorrect_lengths, alternative='two-sided')[1])
p_values.append(mannwhitneyu(correct_sims, incorrect_sims, alternative='greater')[1])
p_values.append(chi2_contingency(contingency)[1])
p_values.append(shapiro(sample)[1])

colors = ['#C44E52' if p < 0.05 else '#55A868' for p in p_values]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(test_names, [-np.log10(p) for p in p_values], color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(-np.log10(0.05), color='red', linestyle='--', linewidth=1.5, label='p=0.05 threshold')
ax.set_xlabel('-log10(p-value)')
ax.set_title('Hypothesis Test Results (red = significant at α=0.05)')
ax.legend()
for bar, p in zip(bars, p_values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'p={p:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/eda_hypothesis_tests.png', bbox_inches='tight')
plt.show()

---
## 9. Position Bias Analysis

In [ ]:
# Does the correct answer tend to be longer or shorter than the other options?
position_data = []
for _, row in df.iterrows():
    lengths = {col: word_count(row[col]) for col in OPTION_COLS}
    sorted_opts = sorted(lengths, key=lengths.get, reverse=True)
    rank = sorted_opts.index(row['answer']) + 1
    position_data.append({'answer': row['answer'], 'length_rank': rank})

pos_df = pd.DataFrame(position_data)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

rank_counts = pos_df['length_rank'].value_counts().sort_index()
axes[0].bar(rank_counts.index, rank_counts.values, color='#4C72B0', edgecolor='black', linewidth=0.5)
axes[0].axhline(len(df)/5, color='red', linestyle='--', label='Expected (uniform)')
axes[0].set_title('Length Rank of Correct Option (1=longest)')
axes[0].set_xlabel('Rank by Length')
axes[0].set_ylabel('Count')
axes[0].legend()

cross = pd.crosstab(pos_df['answer'], pos_df['length_rank'])
sns.heatmap(cross, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            cbar_kws={'shrink': 0.8})
axes[1].set_title('Correct Answer vs Length Rank')
axes[1].set_xlabel('Length Rank of Correct Option')
axes[1].set_ylabel('Answer Label')

plt.tight_layout()
plt.savefig('../reports/eda_position_bias.png', bbox_inches='tight')
plt.show()

---
## 10. EDA Summary

In [ ]:
print('=' * 65)
print('  EDA SUMMARY')
print('=' * 65)
print(f'  Dataset size              : {len(df)} rows')
print(f'  Missing values            : None')
print(f'  Answer distribution       : B > C > A > D > E (non-uniform)')
print(f'  Prompt vocab size         : {len(prompt_vocab):,} unique words')
print(f'  Option vocab size         : {len(option_vocab):,} unique words')
print(f'  Vocab overlap             : {len(overlap_vocab):,} words')
print(f'  Mean prompt length        : {df["prompt_words"].mean():.1f} words')
print(f'  Correct vs incorrect sim  : {np.mean(correct_sims):.4f} vs {np.mean(incorrect_sims):.4f}')
print()
print('  Hypothesis Test Results:')
results = [
    ('Answer non-uniformity (Chi-Sq)',    p_values[0]),
    ('Prompt len by answer (Kruskal)',    p_values[1]),
    ('Option len correct vs not (MWU)',   p_values[2]),
    ('TF-IDF sim correct vs not (MWU)',   p_values[3]),
    ('Answer vs option length (Chi-Sq)',  p_values[4]),
    ('Prompt normality (Shapiro-Wilk)',   p_values[5]),
]
for name, p in results:
    sig = '✅ Significant' if p < 0.05 else '❌ Not significant'
    print(f'  {name:<42}: p={p:.4f}  {sig}')
print('=' * 65)
print('  Plots saved to ../reports/')